In [44]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [45]:
from scikeras.wrappers import KerasClassifier

model = KerasClassifier(model=build_model, verbose=0)

In [46]:
data=pd.read_csv('Churn_Modelling.csv')
data = data.drop(['RowNumber','CustomerId','Surname'],axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder()
geo_encoder = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoder_df = pd.DataFrame(geo_encoder,columns=onehot_encoder_geo.get_feature_names_out(['Geography']))


data = pd.concat([data.drop('Geography',axis=1),geo_encoder_df],axis=1)

x = data.drop('Exited',axis=1)
y = data['Exited']


#split the dataset in traning and testing sets
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)


#scale these features
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)
    
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler, file)



In [47]:
#define function to create model and try different parameters(kerasclasifier)

# def create_model(neurons=32,layers=1):
#     model=Sequential()
#     model.add(Dense(neurons,activation='relu',input_shape =(x_train[1])))
    
#     for _ in range(layers-1):
#         model.add(Dense(neurons,activation='relu'))
        
#     model.add(Dense(1,activation='sigmoid'))
#     model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
#     return model

In [48]:
def build_model():
    model = Sequential()
    
    model.add(Dense(64, activation='relu', input_dim=x_train.shape[1]))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    return model

In [49]:
#create a keras classifier
model=KerasClassifier(layers=-1,neurons=32,build_fn=build_model,epochs=50,batch_size=10,verbose=0)

In [52]:
param_grid = {
    'batch_size': [16, 32],
    'epochs': [50]
}

In [53]:
model = build_model()

model.fit(x_train, y_train, epochs=50, batch_size=32)

Epoch 1/50
250/250 [==============================] - 2s 2ms/step - loss: 0.4585 - accuracy: 0.7991
Epoch 2/50
250/250 [==============================] - 1s 2ms/step - loss: 0.3890 - accuracy: 0.8409
Epoch 3/50
250/250 [==============================] - 0s 2ms/step - loss: 0.3616 - accuracy: 0.8521
Epoch 4/50
250/250 [==============================] - 1s 2ms/step - loss: 0.3507 - accuracy: 0.8550
Epoch 5/50
250/250 [==============================] - 1s 2ms/step - loss: 0.3436 - accuracy: 0.8580
Epoch 6/50
250/250 [==============================] - 1s 3ms/step - loss: 0.3391 - accuracy: 0.8594
Epoch 7/50
250/250 [==============================] - 1s 3ms/step - loss: 0.3370 - accuracy: 0.8619
Epoch 8/50
250/250 [==============================] - 1s 3ms/step - loss: 0.3328 - accuracy: 0.8620
Epoch 9/50
250/250 [==============================] - 1s 3ms/step - loss: 0.3311 - accuracy: 0.8627
Epoch 10/50
250/250 [==============================] - 1s 2ms/step - loss: 0.3292 - accuracy: 0.8656

In [ ]:
# #perform grid search
# grid=GridSearchCV(estimator=model,param_grid=param_grid,n_jobs=-1,cv=3,verbose=1)
# grid_result = grid.fit(x_train,y_train)

print("Best: %f using %s" % (grid_result.best_score_,grid_result.best_params_))

AttributeError: 'Sequential' object has no attribute 'best_score_'